## 1) Pré-processamento: Limpeza dos dados

Primeiramente, temos que obter o dataset e verificar sua integridade.

In [67]:
import pandas as pd


dados = pd.read_csv('_ASSOC_VoleiStars.csv', encoding='latin1')
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Partida         150 non-null    int64  
 1   Jogadore(a)s    150 non-null    str    
 2   Resultado       150 non-null    str    
 3   Unnamed: 3      0 non-null      float64
 4   Unnamed: 4      0 non-null      float64
 5   Jogadore(a)s.1  8 non-null      str    
dtypes: float64(2), int64(1), str(3)
memory usage: 7.2 KB


De acordo com o pdf, 
> O Conjunto é composto pelo número da partida, o nome do(a)s jogadore(a)spresentes, e o resultado da partida.

Contudo, percebe-se que existem no dataframe duas colunas sem nome e sem registro. Seria interessante remove-las para que não poluam o dataset.

In [68]:
dados = dados.drop(columns=['Unnamed: 3', 'Unnamed: 4'])
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Partida         150 non-null    int64
 1   Jogadore(a)s    150 non-null    str  
 2   Resultado       150 non-null    str  
 3   Jogadore(a)s.1  8 non-null      str  
dtypes: int64(1), str(3)
memory usage: 4.8 KB


Existe também uma coluna que contém os nomes dos oito jogadores. Para fins de análise, removeremos essa coluna do dataframe, mas vamos guardar seus valores para limpeza e posterior transformações.

In [69]:
jogadores = dados['Jogadore(a)s.1'].dropna()
dados = dados.drop(columns=['Jogadore(a)s.1'])
jogadores

0    Ricardo
1      Fábio
2      Ágata
3        Ana
4    Bárbara
5     Shelda
6    Sheldom
7    Emanuel
Name: Jogadore(a)s.1, dtype: str

Agora que a tabela contém apenas os dados relevantes, podemos analisar a integridade deles.

In [70]:
dados.head()

,Partida,Jogadore(a)s,Resultado
0,1,"ricardo, fabio, emanuel",Perdeu
1,2,"Emanuel, fabio, ?gata",Perdeu
2,3,"?gata, Sheldom, Ana",GANHOU
3,4,"?gata, Ana, B rbara",GANHOU
4,5,"B rbara, F bio, Ricardo",Perdeu


Percebe-se que claramente o nome dos jogadores em algumas das fileiras está errado. Vamos ver quantas escritas erradas diferentes existem...

In [71]:
nomes_errados = []

for time in dados['Jogadore(a)s']:
    for nome in time.split(','):
        nome = nome.strip()
        if nome not in nomes_errados:
            nomes_errados.append(nome)
            
nomes_errados.sort()
print(f'Número de nomes diferentes encontrados: {len(nomes_errados)}')
nomes_errados

Número de nomes diferentes encontrados: 17


['?gata',
 'Agata',
 'Ana',
 'Barbara',
 'B\xa0rbara',
 'Emanuel',
 'Fabio',
 'F\xa0bio',
 'Ricardo',
 'Shelda',
 'Sheldom',
 'ana',
 'emanuel',
 'fabio',
 'ricardo',
 'shelda',
 'sheldom']

Existem 17 nomes no total, 9 mais do que o esperado. Analisando esses nomes, três erros ficam aparentes:

1. Certos nomes são registrados duas vezes: uma vez em caixa alta, outra completamente minúsculo
2. A letra "Á" maiúscula é substituída por "?"
3. A letra "á" minúscula é substituída por "\xa0"

Dessa forma, a limpeza dos dados errôneos fica fácil. Vamos criar um dicionário que mapeia {nome_errado: nome_correto} para que seja possível transformar os dados posteriormente

In [72]:
map_nome_errado_correto = {}

for nome_errado in nomes_errados:
    # obs: não substituimos "?" por "Á" nem "\xa0" por "á". Isso porque teriamos que tratar strings como "Ágata" e "Agata" como iguais, 
    # o que complicaria o código e não é relevante para fins de extração de padrões.
    nome_correto = nome_errado.capitalize().replace('?', 'A').replace('\xa0', 'a')
    map_nome_errado_correto[nome_errado] = nome_correto

# Pelo motivo mencionado acima, também vamos atualizar a lista dos nomes dos jogadores com os nomes sem acento
jogadores = sorted(list(set(map_nome_errado_correto.values())))
print(f'Nomes corretos: {jogadores}')
print(f'Número de nomes corretos: {len(jogadores)}')

map_nome_errado_correto

Nomes corretos: ['Agata', 'Ana', 'Barbara', 'Emanuel', 'Fabio', 'Ricardo', 'Shelda', 'Sheldom']
Número de nomes corretos: 8


{'?gata': 'Agata',
 'Agata': 'Agata',
 'Ana': 'Ana',
 'Barbara': 'Barbara',
 'B\xa0rbara': 'Barbara',
 'Emanuel': 'Emanuel',
 'Fabio': 'Fabio',
 'F\xa0bio': 'Fabio',
 'Ricardo': 'Ricardo',
 'Shelda': 'Shelda',
 'Sheldom': 'Sheldom',
 'ana': 'Ana',
 'emanuel': 'Emanuel',
 'fabio': 'Fabio',
 'ricardo': 'Ricardo',
 'shelda': 'Shelda',
 'sheldom': 'Sheldom'}

Agora que temos uma estrutura que mapea os nomes errados em nomes corretos, podemos fazer a transformação do dataframe original para que as regras de associação possam ser extraídas.

## 2) Transformação

Agora que temos uma forma de limpar os dados, podemos criar um dataframe com dados limpos e transformados para representação de one-hot encoding.

In [73]:
linhas = []

for indice, linha in dados.iterrows():
    nova_linha = {}
    
    membros_da_equipe = linha['Jogadore(a)s'].split(',')
    for i in range(len(membros_da_equipe)):
        membros_da_equipe[i] = membros_da_equipe[i].strip()
        membros_da_equipe[i] = map_nome_errado_correto[membros_da_equipe[i]]
    
    for jogador in jogadores:
        nova_linha[jogador] = jogador in membros_da_equipe
    
    if linha['Resultado'].lower() == 'ganhou':
        nova_linha['Ganhou'] = True
        nova_linha['Perdeu'] = False
    else:
        nova_linha['Ganhou'] = False
        nova_linha['Perdeu'] = True
        
    linhas.append(nova_linha)
        
volei_stars = pd.DataFrame(linhas)
volei_stars.head(len(volei_stars))


,Agata,Ana,Barbara,Emanuel,Fabio,Ricardo,Shelda,Sheldom,Ganhou,Perdeu
0,False,False,False,True,True,True,False,False,False,True
1,True,False,False,True,True,False,False,False,False,True
2,True,True,False,False,False,False,False,True,True,False
3,True,True,True,False,False,False,False,False,True,False
4,False,False,True,False,True,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...
145,True,True,False,False,False,False,False,False,True,False
146,False,False,False,False,False,True,True,False,False,True
147,True,False,False,False,False,False,False,True,True,False
148,True,True,False,False,False,False,False,False,False,True


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Generate frequent itemsets (using 5% to ensure we capture losing combinations too)
frequent_itemsets = apriori(volei_stars, min_support=0.05, use_colnames=True)

# 2. Generate association rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)

# 3. WINNING COMBINATION: Consequent is 'Ganhou', Support >= 0.10
# Sorted by highest confidence, then highest support
win_rules = rules[
    (rules['consequents'] == frozenset({'Ganhou'})) & 
    (rules['support'] >= 0.10)
].sort_values(by=['confidence', 'support'], ascending=[False, False])

print("--- TOP WINNING COMBINATIONS ---")
print(win_rules[['antecedents', 'support', 'confidence']].head(1))

# 4. LOSING COMBINATION: Consequent is 'Perdeu'from mlxtend.frequent_patterns import apriori, association_rules

# 1. Generate frequent itemsets (using 5% to ensure we capture losing combinations too)
frequent_itemsets = apriori(volei_stars, min_support=0.05, use_colnames=True)

# 2. Generate association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

# 3. WINNING COMBINATION: Consequent is 'Ganhou', Support >= 0.10
# Sorted by highest confidence, then highest support
win_rules = rules[
    (rules['consequents'] == frozenset({'Ganhou'})) & 
    (rules['support'] >= 0.10)
].sort_values(by=['confidence', 'support'], ascending=[False, False])

print("--- TOP WINNING COMBINATIONS ---")
print(win_rules[['antecedents', 'support', 'confidence']].head(1))

# 4. LOSING COMBINATION: Consequent is 'Perdeu'
# Sorted by highest confidence to see who guarantees a loss
lose_rules = rules[
    (rules['consequents'] == frozenset({'Perdeu'}))
].sort_values(by=['confidence', 'support'], ascending=[False, False])

print("\n--- TOP LOSING COMBINATIONS ---")
print(lose_rules[['antecedents', 'support', 'confidence']].head(1))

# 5. THE STAR: Single player implying 'Ganhou' with the highest confidence
# Filters the winning rules down to itemsets containing exactly 1 player
single_player_win = win_rules[win_rules['antecedents'].apply(lambda x: len(x) == 1)]

print("\n--- THE STAR (Best Individual Performer) ---")
print(single_player_win[['antecedents', 'support', 'confidence']].head(1))
# Sorted by highest confidence to see who guarantees a loss
lose_rules = rules[
    (rules['consequents'] == frozenset({'Perdeu'}))
].sort_values(by=['confidence', 'support'], ascending=[False, False])

print("\n--- TOP LOSING COMBINATIONS ---")
print(lose_rules[['antecedents', 'support', 'confidence']].head(1))

# 5. THE STAR: Single player implying 'Ganhou' with the highest confidence
# Filters the winning rules down to itemsets containing exactly 1 player
single_player_win = win_rules[win_rules['antecedents'].apply(lambda x: len(x) == 1)]

print("\n--- THE STAR (Best Individual Performer) ---")
print(single_player_win[['antecedents', 'support', 'confidence']].head(1))

--- TOP WINNING COMBINATIONS ---
                  antecedents   support  confidence
16  frozenset({Ana, Barbara})  0.113333        0.68
--- TOP WINNING COMBINATIONS ---
                  antecedents   support  confidence
16  frozenset({Ana, Barbara})  0.113333    0.680000
12    frozenset({Ana, Agata})  0.140000    0.677419
2            frozenset({Ana})  0.300000    0.661765

--- TOP LOSING COMBINATIONS ---
                      antecedents   support  confidence
25   frozenset({Ricardo, Shelda})  0.060000    0.750000
26  frozenset({Sheldom, Ricardo})  0.060000    0.692308
9             frozenset({Shelda})  0.133333    0.625000

--- THE STAR (Best Individual Performer) ---
        antecedents  support  confidence
2  frozenset({Ana})      0.3    0.661765

--- TOP LOSING COMBINATIONS ---
                     antecedents  support  confidence
25  frozenset({Ricardo, Shelda})     0.06        0.75

--- THE STAR (Best Individual Performer) ---
        antecedents  support  confidence
2  frozen